In [0]:
from pyspark.sql.functions import *

final_df = spark.table("silver_healthcare")

display(final_df)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500.0,6500.0,6,Adult
D102,P102,A1002,2026-06-01,Migraine,3500.0,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000.0,5500.0,6,Young
D103,P103,A1003,2026-06-02,Skin Allergy,2000.0,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000.0,3000.0,6,Adult
D104,P104,A1004,2026-06-02,Fracture,12000.0,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,14500.0,6,Adult
D105,P105,A1005,2026-06-03,Fever,1500.0,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200.0,2700.0,6,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000.0,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000.0,10000.0,6,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500.0,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500.0,7000.0,6,Young
D103,P108,A1008,2026-06-04,Skin Infection,2500.0,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000.0,3500.0,6,Adult
D106,P101,A1009,2026-06-05,Cardiac Review,6500.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000.0,9500.0,6,Adult
D104,P104,A1010,2026-06-05,Back Pain,4500.0,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,7000.0,6,Adult


In [0]:
dept_revenue = final_df.groupBy(
    "department"
).agg(
    sum("final_bill").alias("revenue")
)

display(dept_revenue)

department,revenue
Cardiology,33000.0
Neurology,5500.0
Dermatology,6500.0
Orthopedics,21500.0
Pediatrics,2700.0


Databricks visualization. Run in Databricks to view.

In [0]:
city_revenue = final_df.groupBy(
    "city"
).agg(
    sum("final_bill").alias("revenue")
)

display(city_revenue)

city,revenue
Hyderabad,23000.0
Bangalore,5500.0
Mumbai,3000.0
Delhi,21500.0
Chennai,2700.0
Pune,10000.0
Kochi,3500.0


Databricks visualization. Run in Databricks to view.

In [0]:
status_df = final_df.groupBy(
    "status"
).count()

display(status_df)

status,count
Completed,7
Pending,2
Cancelled,1


Databricks visualization. Run in Databricks to view.

In [0]:
doctor_revenue = final_df.groupBy(
    "doctor_name"
).agg(
    sum("final_bill").alias("revenue")
).orderBy(desc("revenue"))

display(doctor_revenue)

doctor_name,revenue
Dr. Suresh,21500.0
Dr. Kiran,19500.0
Dr. Ramesh,13500.0
Dr. Anita,6500.0
Dr. Priya,5500.0
Dr. Meera,2700.0


Databricks visualization. Run in Databricks to view.

In [0]:
revenue_trend = final_df.groupBy(
    "appointment_date"
).agg(
    sum("final_bill").alias("revenue")
).orderBy("appointment_date")

display(revenue_trend)

appointment_date,revenue
2026-06-01,12000.0
2026-06-02,17500.0
2026-06-03,12700.0
2026-06-04,10500.0
2026-06-05,16500.0


In [0]:
%sql
CREATE TABLE managed_healthcare
AS
SELECT * FROM silver_healthcare;

num_affected_rows,num_inserted_rows


In [0]:
final_df.write.format("delta") \
.mode("overwrite") \
.save("/Volumes/databricks_akshaya/default/healthcare/external_healthcare")

In [0]:
%sql
CREATE TABLE external_healthcare
USING DELTA
LOCATION '/Volumes/databricks_akshaya/default/healthcare/external_healthcare';

---------------------------------------------------------------------------
AnalysisException Traceback (most recent call last)
File , line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE TABLE external_healthcare\nUSING DELTA\nLOCATION '/Volumes/databricks_akshaya/default/healthcare/external_healthcare';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
 2539 with self.builtin_trap:
 2540 args = (magic_arg_s, cell)
-> 2541 result = fn(*args, **kwargs)
 2543 # The code below prevents the output from being displayed
 2544 # when using magics with decorator @output_can_be_silenced
 2545 # when the last Python token in the expression is a ';'.
 2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
 206 except BaseException as e:
 207 self.driver_activity_logger.logExecuteCommandEvent(
 208 "SQL_MAGIC_PY4J_FAILED",
 209 exceptionClassName=e.__class__.__name__,
 210 sqlState=safe_call(e, "getSqlState"),
 211 errorClass=safe_call(e, "getCondition"),
 212 )
--> 213 raise e
 215 self.driver_activity_logger.logExecuteCommandEvent("SQL_MAGIC_PY4J_SUCCEEDED")
 216 return result

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:205, in SqlMagic.sql(self, line, cell)
 203 query_text = sub_query.query()
 204 sql_directive = self._execution_lifecycle_client.getSqlDirective(query_text)
--> 205 result = self._handle_sql_directive(sql_directive, i == number_of_sub_queries - 1)
 206 except BaseException as e:
 207 self.driver_activity_logger.logExecuteCommandEvent(
 208 "SQL_MAGIC_PY4J_FAILED",
 209 exceptionClassName=e.__class__.__name__,
 210 sqlState=safe_call(e, "getSqlState"),
 211 errorClass=safe_call(e, "getCondition"),
 212 )

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:227, in SqlMagic._handle_sql_directive(self, sql_directive, is_last_query)
 225 df = self._get_query_request_result(query)
 226 elif directive_name in ("CreateView", "DropTable", "AlterTable", "CreateTable"):
--> 227 self._get_query_request_result(sql_directive.sql())
 228 elif directive_name == "CacheTableAs":
 229 table_name = sql_directive.table()

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:168, in SqlMagic._get_query_request_result(self, query)
 166 if len(widget_bindings := self._widget_cache.values) > 0:
 167 self.driver_activity_logger.logExecuteCommandEvent("PARAM_SYNTAX_USAGE")
--> 168 df = self.spark.sql(query, widget_bindings)
 169 return df

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
 898 _views.append(SubqueryAlias(df._plan, name))
 900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
 902 if "sql_command_result" in properties:
 903 df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
 1536 req.user_context.user_id = self._user_id
 1537 self._set_command_in_plan(req.plan, command)
-> 1538 data, _, metrics, observed_metrics, properties = self._execute_and_fetch(
 1539 req, observations or {}, extra_request_metadata
 1540 )
 1541 # Create a query execution object.
 1542 ei = ExecutionInfo(metrics, observed_metrics)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:2086, in SparkConnectClient._execute_and_fetch(self, req, observations, extra_request_metadata, self_destruct)
 2083 properties: Dict[str, Any] = {}
 2085 with Progress(handlers=self._progress_handlers, operation_id=req.operation_id) as progress:
-> 2086 for response in self._execute_a

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_healthcare AS
SELECT * FROM silver_healthcare;

In [0]:
%sql
SELECT * FROM temp_healthcare;

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500.0,6500.0,6,Adult
D102,P102,A1002,2026-06-01,Migraine,3500.0,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000.0,5500.0,6,Young
D103,P103,A1003,2026-06-02,Skin Allergy,2000.0,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000.0,3000.0,6,Adult
D104,P104,A1004,2026-06-02,Fracture,12000.0,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,14500.0,6,Adult
D105,P105,A1005,2026-06-03,Fever,1500.0,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200.0,2700.0,6,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000.0,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000.0,10000.0,6,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500.0,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500.0,7000.0,6,Young
D103,P108,A1008,2026-06-04,Skin Infection,2500.0,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000.0,3500.0,6,Adult
D106,P101,A1009,2026-06-05,Cardiac Review,6500.0,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000.0,9500.0,6,Adult
D104,P104,A1010,2026-06-05,Back Pain,4500.0,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500.0,7000.0,6,Adult
